**Polars**

In [ ]:
import polars as pl

# 1. Чтение датасета
df = pl.read_csv("train.csv")

print("Первые строки датасета:")
print(df.head())

# 2. Основная информация о датасете
print("\nРазмер датасета (строки, столбцы):")
print(df.shape)

print("\nТипы данных:")
print(df.dtypes)

print("\nКоличество пропусков по столбцам:")
print(df.null_count())

print("\nОписательная статистика:")
print(df.describe())

print("\nСредние значения числовых столбцов:")
print(
    df.select(
        pl.col("PassengerId"),
        pl.col("Survived"),
        pl.col("Pclass"),
        pl.col("Age"),
        pl.col("SibSp"),
        pl.col("Parch"),
        pl.col("Fare"),
    ).mean()
)


# 3. Количество пассажиров каждого класса
print("\nКоличество пассажиров каждого класса (Pclass):")
pclass_counts = (
    df.get_column("Pclass")
    .value_counts()
    .sort("Pclass")
)
print(pclass_counts)

# 4. Количество выживших мужчин и женщин
print("\nКоличество выживших мужчин и женщин:")
survived_by_sex = (
    df.group_by("Sex")
    .agg(
        pl.col("Survived").sum().alias("survived_count"),
        pl.len().alias("total_passengers"),
    )
    .sort("Sex")
)
print(survived_by_sex)


# 5. Пассажиры старше 44 лет
print("\nПассажиры старше 44 лет:")
passengers_over_44 = df.filter(pl.col("Age") > 44)
print(passengers_over_44)

print(f"\nВсего пассажиров старше 44 лет: {passengers_over_44.height}")

Первые строки датасета:
shape: (5, 12)
┌─────────────┬──────────┬────────┬───────────────────┬───┬───────────┬─────────┬───────┬──────────┐
│ PassengerId ┆ Survived ┆ Pclass ┆ Name              ┆ … ┆ Ticket    ┆ Fare    ┆ Cabin ┆ Embarked │
│ ---         ┆ ---      ┆ ---    ┆ ---               ┆   ┆ ---       ┆ ---     ┆ ---   ┆ ---      │
│ i64         ┆ i64      ┆ i64    ┆ str               ┆   ┆ str       ┆ f64     ┆ str   ┆ str      │
╞═════════════╪══════════╪════════╪═══════════════════╪═══╪═══════════╪═════════╪═══════╪══════════╡
│ 1           ┆ 0        ┆ 3      ┆ Braund, Mr. Owen  ┆ … ┆ A/5 21171 ┆ 7.25    ┆ null  ┆ S        │
│             ┆          ┆        ┆ Harris            ┆   ┆           ┆         ┆       ┆          │
│ 2           ┆ 1        ┆ 1      ┆ Cumings, Mrs.     ┆ … ┆ PC 17599  ┆ 71.2833 ┆ C85   ┆ C        │
│             ┆          ┆        ┆ John Bradley (Fl… ┆   ┆           ┆         ┆       ┆          │
│ 3           ┆ 1        ┆ 3      ┆ Heikkinen, Miss.

**Ускорение работы с pandas**

In [ ]:
import pandas as pd
import bottleneck as bn

df = pd.read_csv('train.csv')


df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
mean_age = bn.nanmean(df['Age'])
std_age = bn.nanstd(df['Age'])

print(f"Средний возраст: {mean_age:.2f}")
print(f"Стандартное отклонение: {std_age:.2f}")

df['Fare_new'] = df['Fare'] * 1.3

print(df[['Fare', 'Fare_new']].head())

Средний возраст: 29.70
Стандартное отклонение: 14.52
      Fare  Fare_new
0   7.2500   9.42500
1  71.2833  92.66829
2   7.9250  10.30250
3  53.1000  69.03000
4   8.0500  10.46500


**Оптимизация типов pandas**

In [ ]:
import pandas as pd

# 1. Загрузка данных
df = pd.read_csv('Housing.csv')

# Статистика по столбцам
print("\nОписание датасета:")
print(df.describe(include='all'))

# Анализ текущего использования памяти
original_memory = df.memory_usage(deep=True).sum()
print(f"Исходное использование памяти: {original_memory} байт")

# 2. Анализ и оптимизация типов данных
# price: int64 -> int32 (максимум ~13.3M, помещается в int32)
# area: int64 -> int16 (максимум 16200, помещается в int16)
# bedrooms: int64 -> int8 (максимум 6 спален)
# bathrooms: int64 -> int8 (максимум 4 ванных)
# stories: int64 -> int8 (максимум 4 этажа)
# parking: int64 -> int8 (максимум 3 парковочных места)
# категориальные признаки -> category

# 3. Применение оптимизаций
# Числовые столбцы
df['price'] = df['price'].astype('int32')  # Экономия с int64 на int32
df['area'] = df['area'].astype('int16')    # Экономия с int64 на int16
df['bedrooms'] = df['bedrooms'].astype('int8')  # Экономия с int64 на int8
df['bathrooms'] = df['bathrooms'].astype('int8')  # Экономия с int64 на int8
df['stories'] = df['stories'].astype('int8')  # Экономия с int64 на int8
df['parking'] = df['parking'].astype('int8')  # Экономия с int64 на int8

# Категориальные признаки
categorical_cols = ['mainroad', 'guestroom', 'basement',
                   'hotwaterheating', 'airconditioning',
                   'prefarea', 'furnishingstatus']

for col in categorical_cols:
    df[col] = df[col].astype('category')

# 4. Сравнение потребления памяти
optimized_memory = df.memory_usage(deep=True).sum()
memory_saving = original_memory - optimized_memory

# 5. Вывод результатов
print("\nИсходный размер датасета:")
print(df.memory_usage(deep=True))
print(f"\nОбщий исходный объем: {original_memory} байт")

print("\nОптимизированный размер датасета:")
print(df.memory_usage(deep=True))
print(f"\nОбщий оптимизированный объем: {optimized_memory} байт")
print(f"Экономия памяти: {memory_saving} байт ({memory_saving/original_memory*100:.2f}%)")

# Дополнительная информация
print("\nОптимизированные типы данных:")
print(df.dtypes)



Описание датасета:
               price          area    bedrooms   bathrooms     stories  \
count   5.450000e+02    545.000000  545.000000  545.000000  545.000000   
unique           NaN           NaN         NaN         NaN         NaN   
top              NaN           NaN         NaN         NaN         NaN   
freq             NaN           NaN         NaN         NaN         NaN   
mean    4.766729e+06   5150.541284    2.965138    1.286239    1.805505   
std     1.870440e+06   2170.141023    0.738064    0.502470    0.867492   
min     1.750000e+06   1650.000000    1.000000    1.000000    1.000000   
25%     3.430000e+06   3600.000000    2.000000    1.000000    1.000000   
50%     4.340000e+06   4600.000000    3.000000    1.000000    2.000000   
75%     5.740000e+06   6360.000000    3.000000    2.000000    2.000000   
max     1.330000e+07  16200.000000    6.000000    4.000000    4.000000   

       mainroad guestroom basement hotwaterheating airconditioning  \
count       545      